In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset

# Setting for generating dataset

In [2]:
# Load the network for Case14
net = {
    "baseMVA": 100.0,
## area data
    "areas": np.array([[1, 4]]),
## bus data
###	bus_i	type	Pd	Qd	Gs	Bs	area	Vm	Va	baseKV	zone	Vmax	Vmin
    "bus": np.array([
                    [1,	 3,	 0.0,	 0.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[2,	 2,	 21.7,	 12.7,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,    0.94000],
                	[3,	 2,	 94.2,	 19.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,    0.94000],
                	[4,	 1,	 47.8,	 -3.9,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[5,  1,	 7.6,	 1.6,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[6,	 2,	 11.2,	 7.5,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[7,	 1,	 0.0,	 0.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[8,	 2,	 0.0,	 0.0,	 0.0,	 0.0,	 1,     1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[9,	 1,	 29.5,	 16.6,	 0.0,	 19.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,    0.94000],
                	[10, 1,	 9.0,	 5.8,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[11, 1,	 3.5,	 1.8,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[12, 1,	 6.1,	 1.6,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[13, 1,	 13.5,	 5.8,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                	[14, 1,	 14.9,	 5.0,	 0.0,	 0.0,	 1,	    1.00000,  0.00000,	 1.0,	 1,	    1.06000,	0.94000],
                ]),
    
## generator data
###	bus	Pg	Qg	Qmax	Qmin	Vg	mBase	status	Pmax	Pmin
    "gen": np.array([
                	[1,	 170.0,	 5.0,	 10.0,	  0.0,	 1.0,	 100.0,	 1,	 340,  0.0],
                	[2,	 29.5,	 0.0,	 30.0,	 -30.0,	 1.0,	 100.0,	 1,	 59,   0.0],
                	[3,	 0.0,	 20.0,	 40.0,	  0.0,	 1.0,	 100.0,	 1,	 0,	   0.0],
                	[6,	 0.0,	 9.0,	 24.0,	 -6.0,	 1.0,	 100.0,	 1,	 0,	   0.0],
                	[8,	 0.0,	 9.0,	 24.0,	 -6.0,	 1.0,	 100.0,	 1,	 0,	   0.0]
                 ]),
    
## generator cost data
###	2	startup	shutdown	n	c(n-1)	...	c0
    "gencost": np.array([
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   7.920951,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	  23.269494,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   0.000000,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   0.000000,	   0.000000],
                	[2,	 0.0,	 0.0,	 3,	   0.000000,	   0.000000,	   0.000000]
                ]),
    
## branch data
###	fbus	tbus	r	x	b	rateA	rateB	rateC	ratio	angle	status	angmin	angmax
    "branch": np.array([
                    [1,	 2,	 0.01938,	 0.05917,	 0.0528, 472,	 472,	 472,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[1,	 5,	 0.05403,	 0.22304,	 0.0492, 128,	 128,	 128,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[2,	 3,	 0.04699,	 0.19797,	 0.0438, 145,	 145,	 145,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                    [2,	 4,	 0.05811,	 0.17632,	 0.034,	 158,	 158,	 158,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[2,	 5,	 0.05695,	 0.17388,	 0.0346, 161,	 161,	 161,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[3,	 4,	 0.06701,	 0.17103,	 0.0128, 160,	 160,	 160,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[4,	 5,	 0.01335,	 0.04211,	 0.0,	 664,	 664,	 664,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[4,	 7,	 0.0,	     0.20912,	 0.0,	 141,	 141,	 141,	 0.978,	 0.0,	 1,	 -30.0,	 30.0],
                	[4,	 9,	 0.0,	     0.55618,	 0.0,	 53,	 53,	 53,	 0.969,	 0.0,	 1,	 -30.0,	 30.0],
                	[5,  6,	 0.0,	     0.25202,	 0.0,	 117,	 117,	 117,	 0.932,	 0.0,	 1,	 -30.0,	 30.0],
                	[6,	 11, 0.09498,	 0.1989,     0.0,	 134,	 134,	 134,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
               	    [6,	 12, 0.12291,	 0.25581,	 0.0,	 104,	 104,	 104,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[6,	 13, 0.06615,	 0.13027,	 0.0,	 201,	 201,	 201,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[7,	 8,	 0.0,	     0.17615,	 0.0,	 167,	 167,	 167,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[7,	 9,	 0.0,	     0.11001,	 0.0,	 267,	 267,	 267,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[9,	 10, 0.03181,	 0.0845,     0.0,	 325,	 325,	 325,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[9,	 14, 0.12711,	 0.27038,	 0.0,	 99,	 99,	 99,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[10, 11, 0.08205,	 0.19207,	 0.0,	 141,	 141,	 141,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[12, 13, 0.22092,	 0.19988,	 0.0,	 99,	 99,	 99,	 0.0,	 0.0,	 1,	 -30.0,	 30.0],
                	[13, 14, 0.17093,	 0.34802,	 0.0,	 76,	 76,	 76,     0.0,	 0.0,	 1,	 -30.0,	 30.0]
                ])
}

In [3]:
print("Type of net:", type(net))

Type of net: <class 'dict'>


In [3]:
cost_gen = pd.DataFrame(net['gencost'], columns = ['2', 'startup', 'shutdown', '3','n', 'c(n-1)', 'c0'])
cost_gen

,2,startup,shutdown,3,n,c(n-1),c0
0,2.0,0.0,0.0,3.0,0.0,7.920951,0.0
1,2.0,0.0,0.0,3.0,0.0,23.269494,0.0
2,2.0,0.0,0.0,3.0,0.0,0.000000,0.0
3,2.0,0.0,0.0,3.0,0.0,0.000000,0.0
4,2.0,0.0,0.0,3.0,0.0,0.000000,0.0


In [4]:
gencost_array = cost_gen[["c(n-1)"]].to_numpy()

print("Gencost array:")
print(gencost_array)

Gencost array:
[[ 7.920951]
 [23.269494]
 [ 0.      ]
 [ 0.      ]
 [ 0.      ]]


In [5]:
line = pd.DataFrame(net['branch'],
                    columns = ['fbus',	'tbus',	'r',	'x',	'b',
                               'rateA',	'rateB',	'rateC',	'ratio',
                               'angle',	'status',	'angmin',	'angmax'] )
line

,fbus,tbus,r,x,b,rateA,rateB,rateC,ratio,angle,status,angmin,angmax
0,1.0,2.0,0.01938,0.05917,0.0528,472.0,472.0,472.0,0.000,0.0,1.0,-30.0,30.0
1,1.0,5.0,0.05403,0.22304,0.0492,128.0,128.0,128.0,0.000,0.0,1.0,-30.0,30.0
2,2.0,3.0,0.04699,0.19797,0.0438,145.0,145.0,145.0,0.000,0.0,1.0,-30.0,30.0
3,2.0,4.0,0.05811,0.17632,0.0340,158.0,158.0,158.0,0.000,0.0,1.0,-30.0,30.0
4,2.0,5.0,0.05695,0.17388,0.0346,161.0,161.0,161.0,0.000,0.0,1.0,-30.0,30.0
5,3.0,4.0,0.06701,0.17103,0.0128,160.0,160.0,160.0,0.000,0.0,1.0,-30.0,30.0
6,4.0,5.0,0.01335,0.04211,0.0000,664.0,664.0,664.0,0.000,0.0,1.0,-30.0,30.0
7,4.0,7.0,0.00000,0.20912,0.0000,141.0,141.0,141.0,0.978,0.0,1.0,-30.0,30.0
8,4.0,9.0,0.00000,0.55618,0.0000,53.0,53.0,53.0,0.969,0.0,1.0,-30.0,30.0
9,5.0,6.0,0.00000,0.25202,0.0000,117.0,117.0,117.0,0.932,0.0,1.0,-30.0,30.0


In [6]:
num_buses = len(net['bus'])

In [7]:
"""Initialize the admittance matrix"""

Y = np.zeros((num_buses, num_buses), dtype=complex)

# Populate the admittance matrix
for _, row in line.iterrows():
    fbus, tbus, r, x, b = int(row['fbus']) - 1, int(row['tbus']) - 1, row['r'], row['x'], row['b']
    y = 1 / (r + 1j * x)  # Line admittance
    Y[fbus, tbus] -= y
    Y[tbus, fbus] -= y  # Symmetric off-diagonal
    Y[fbus, fbus] += y + 1j * b / 2  # Diagonal element for from-bus
    Y[tbus, tbus] += y + 1j * b / 2  # Diagonal element for to-bus

# Separate into G (conductance) and B (susceptance)
G_matrix = Y.real
B_matrix = Y.imag


# Convert to DataFrames for better visualization
G_df = pd.DataFrame(G_matrix, columns=[f"Bus {i+1}" for i in range(num_buses)], index=[f"Bus {i+1}" for i in range(num_buses)])
B_df = pd.DataFrame(B_matrix, columns=[f"Bus {i+1}" for i in range(num_buses)], index=[f"Bus {i+1}" for i in range(num_buses)])

# print("Conductance Matrix (G):")
# print(G_df)

# print("\nSusceptance Matrix (B):")
# print(B_df)

# Ouput from MatPower

In [8]:
df = pd.read_csv('./data/pglib_opf_case14_ieee.csv')
df.head()

,load1:pl,load2:pl,load3:pl,load4:pl,load5:pl,load6:pl,load7:pl,load8:pl,load9:pl,load10:pl,...,line11:q_fr_max,line12:q_fr_max,line13:q_fr_max,line14:q_fr_max,line15:q_fr_max,line16:q_fr_max,line17:q_fr_max,line18:q_fr_max,line19:q_fr_max,line20:q_fr_max
0,0.520608,0.222042,0.449614,0.218480,0.202422,0.404573,0.091053,0.223092,0.292456,0.259478,...,0.0,0.0,0.0,-2.021554e-07,-4.062060e-08,0.000000e+00,0.0,0.000000e+00,0.0,-1.518648e-07
1,0.573892,0.299027,0.358778,0.196437,0.222381,0.417727,0.109823,0.162996,0.410181,0.255933,...,0.0,0.0,0.0,-3.546603e-07,-3.537620e-08,-6.407696e-09,0.0,-3.914912e-08,0.0,0.000000e+00
2,0.419371,0.430017,0.416764,0.156606,0.370736,0.334658,0.151944,0.111594,0.387565,0.208321,...,0.0,0.0,0.0,-3.722503e-07,-2.438078e-09,-1.013408e-08,0.0,-5.826317e-08,0.0,0.000000e+00
3,0.594419,0.355337,0.352113,0.188077,0.209334,0.438329,0.155785,0.110417,0.432763,0.211087,...,0.0,0.0,0.0,-3.539681e-07,-2.845422e-08,-1.096959e-08,0.0,-6.006438e-08,0.0,0.000000e+00
4,0.294095,0.287025,0.484957,0.260890,0.352300,0.378632,0.038344,0.283561,0.213451,0.205223,...,0.0,0.0,0.0,-1.705868e-07,-5.639175e-08,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00


# Funtion to generate the tensor dataset based on the input data

In [9]:
def generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix, gencost, num_cost_coefficients=1):
    """
    Generate a tensor dataset following the specified format.

    Args:
        data (pd.DataFrame): DataFrame containing the sample data for loads, generation, and branch information.
        num_buses (int): Number of buses (N_B).
        num_generators (int): Number of generators (N_G).
        G_matrix (np.array): Conductance matrix
        B_matrix (np.array): Susceptance matrix
        gencost (np.array): Generator cost data matrix
        num_cost_coefficients (int): Number of cost coefficients (N_C) # 0. Default is 1 for 1st-order polynomial.

    Returns:
        np.ndarray: Tensor dataset of shape [N_I, (4 + N_C * N_G), N_B, N_B].
    """
    # Extract column mappings
    pd_columns = [col for col in data.columns if "load" in col and ":pl" in col]  # Active power loads (P_d)
    qd_columns = [col for col in data.columns if "load" in col and ":ql" in col]  # Reactive power loads (Q_d)
    num_samples = len(data)
    num_channels = 4 + (num_generators * num_cost_coefficients)  # 4 primary channels + cost channels

    # Initialize the tensor dataset
    tensor_dataset = np.zeros((num_samples, num_channels, num_buses, num_buses))

    # Process each sample
    for sample_idx in range(num_samples):
        sample = data.iloc[sample_idx]

        # Step 1: Active and Reactive Power Demand Matrices (P_d and Q_d)
        Pd_matrix = np.zeros((num_buses, num_buses))
        Qd_matrix = np.zeros((num_buses, num_buses))
        Pd_matrix[np.diag_indices(len(pd_columns))] = sample[pd_columns].values
        Qd_matrix[np.diag_indices(len(qd_columns))] = sample[qd_columns].values

        # Step 2: Placeholder Admittance Matrices (G and B)
        G_matrix = G_matrix
        B_matrix = B_matrix

        # Step 3: Cost Matrices for Generation Costs
        cost_matrices = []
        for gen_idx in range(num_generators):
            cost_matrix = np.zeros((num_buses, num_buses))
            
            # Extract cost efficients from gencost
            coefficient = gencost[gen_idx, 0]

            # Populate digonal with the generation cost coefficient
            cost_matrix[gen_idx, gen_idx] = coefficient
                
            cost_matrices.append(cost_matrix)
            
        # Combine all channels into a tensor for this sample
        sample_tensor = [Pd_matrix, Qd_matrix, G_matrix, B_matrix] + cost_matrices
        tensor_dataset[sample_idx] = np.stack(sample_tensor, axis=0)

    return tensor_dataset

In [10]:
file_path = './data/pglib_opf_case14_ieee.csv'
data = pd.read_csv(file_path)

In [11]:
num_buses = 14
num_generators = 5

In [12]:
tensor_dataset = generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix,gencost=gencost_array)

In [13]:
tensor_dataset.shape

(10000, 9, 14, 14)

In [14]:
#tensor_dataset[0][4]

# Define dataset for training 

In [15]:
def generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix, gencost, num_cost_coefficients=1):
    """
    Generate a tensor dataset following the specified format.

    Args:
        data (pd.DataFrame): DataFrame containing the sample data for loads, generation, and branch information.
        num_buses (int): Number of buses (N_B).
        num_generators (int): Number of generators (N_G).
        G_matrix (np.array): Conductance matrix
        B_matrix (np.array): Susceptance matrix
        gencost (np.array): Generator cost data matrix
        num_cost_coefficients (int): Number of cost coefficients (N_C). Default is 1 for 1st-order polynomial.

    Returns:
        pd.DataFrame: Updated DataFrame with new calculated columns.
        np.ndarray: Tensor dataset of shape [N_I, (4 + N_C * N_G), N_B, N_B].
    """
    df = data.copy()
    
    """==================================Output============================================="""
    # Define outputs
    gen_p = [col for col in df.columns if col.endswith(":pg")]
    gen_q = [col for col in df.columns if col.endswith(":qg")]

    def convert(x):
        if isinstance(x, str):
            x = x.replace(" + ", "+").replace(" - ", "-").replace(" j ", "j").strip()
            return np.complex64(x)
        else:
            return np.complex64(0)

    bus_vm = []
    bus_va = []
    for bus_v_column in [col for col in df.columns if ":v_bus" in col]:
        df[bus_v_column + '_mag'] = df[bus_v_column].apply(convert).apply(np.abs)          # calculate the magnitude
        bus_vm.append(bus_v_column + '_mag')

        df[bus_v_column + '_ang'] = df[bus_v_column].apply(convert).apply(np.angle)        # calculate the angle
        df[bus_v_column + '_ang'] = -1 * np.rad2deg(df[bus_v_column + '_ang'].values)      # convert to radian
        bus_va.append(bus_v_column + '_ang')
        
    # Define outputs
    outputs = gen_p + gen_q + bus_vm + bus_va

    """==================================Input============================================="""
    # Extract column mappings
    pd_columns = [col for col in data.columns if "load" in col and ":pl" in col]  # Active power loads (P_d)
    qd_columns = [col for col in data.columns if "load" in col and ":ql" in col]  # Reactive power loads (Q_d)
    num_samples = len(data)
    num_channels = 4 + (num_generators * num_cost_coefficients)  # 4 primary channels + cost channels

    # Initialize the tensor dataset
    tensor_dataset = np.zeros((num_samples, num_channels, num_buses, num_buses))

    # Process each sample
    for sample_idx in range(num_samples):
        sample = data.iloc[sample_idx]

        # Step 1: Active and Reactive Power Demand Matrices (P_d and Q_d)
        Pd_matrix = np.zeros((num_buses, num_buses))
        Qd_matrix = np.zeros((num_buses, num_buses))
        Pd_matrix[np.diag_indices(len(pd_columns))] = sample[pd_columns].values
        Qd_matrix[np.diag_indices(len(qd_columns))] = sample[qd_columns].values

        # Step 2: Admittance Matrices (G and B)
        G_matrix = G_matrix
        B_matrix = B_matrix

        # Step 3: Cost Matrices for Generation Costs
        cost_matrices = []
        for gen_idx in range(num_generators):
            cost_matrix = np.zeros((num_buses, num_buses))

            # Extract cost coefficients from gencost
            coefficient = gencost[gen_idx, 0]

            # Populate diagonal with the generation cost coefficient
            cost_matrix[gen_idx, gen_idx] = coefficient

            cost_matrices.append(cost_matrix)

        # Combine all channels into a tensor for this sample
        sample_tensor = [Pd_matrix, Qd_matrix, G_matrix, B_matrix] + cost_matrices
        tensor_dataset[sample_idx] = np.stack(sample_tensor, axis=0)

    return df[outputs], tensor_dataset


In [16]:
class OPFDataset(Dataset):
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        input_tensor = torch.tensor(self.inputs[idx]).to(torch.float32)
        output_tensor = torch.tensor(self.outputs[idx]).to(torch.float32)
        return input_tensor, output_tensor
        

In [17]:
outputs, inputs = generate_tensor_dataset(data, num_buses, num_generators, G_matrix, B_matrix, gencost=gencost_array)


In [18]:
outputs.shape

(10000, 38)

In [19]:
inputs.shape

(10000, 9, 14, 14)

In [20]:
inputs[0][0]

array([[0.52060824, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.22204189, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.44961395, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.21848019, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.20242186,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0. 

In [21]:
array = inputs[0][0]

In [22]:
non_zero_values = array[array != 0]
non_zero_values

array([0.52060824, 0.22204189, 0.44961395, 0.21848019, 0.20242186,
       0.40457264, 0.09105288, 0.22309238, 0.29245577, 0.25947801,
       0.13045342])

In [23]:
array_1 = inputs[0][1]

In [24]:
non_zero_values_1 = array_1[array_1 != 0]
non_zero_values_1

array([0.00407739, 0.06743897, 0.21115388, 0.04736138, 0.15619262,
       0.03810965, 0.10610487, 0.00170097])

In [25]:
a = inputs[0][0][inputs[0][0] != 0]
a

array([0.52060824, 0.22204189, 0.44961395, 0.21848019, 0.20242186,
       0.40457264, 0.09105288, 0.22309238, 0.29245577, 0.25947801,
       0.13045342])

In [26]:
X_train, X_test, y_train, y_test = train_test_split(tensor_dataset, outputs.values, test_size=0.2, random_state=42)

In [27]:
train_dataset = OPFDataset(X_train, y_train)
test_dataset = OPFDataset(X_test, y_test)

In [28]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Define model

In [29]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import os
import random
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import trange, tqdm

In [30]:
class CNN(nn.Module):
    def __init__(self, channels_in, y_size):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(channels_in, 64, 3, 1, 1, padding_mode='reflect')

        self.norm = nn.LayerNorm(64)
        self.mha  = nn.MultiheadAttention(64, num_heads=1, batch_first=True)
        self.scale = nn.Parameter(torch.zeros(1))

        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(128)

        self.drop = nn.Dropout(0.5)
        self.fc_1 = nn.Linear(128*14*14, 256)
        self.fc_out = nn.Linear(256, y_size)

    def use_attention(self, x):
        bs, c, h, w = x.shape
        x_attn = x.reshape(bs, c, h * w).transpose(1, 2)    # BS X HW X C

        x_attn = self.norm(x_attn)
        att_out, att_map = self.mha(x_attn, x_attn, x_attn)
        return att_out.transpose(1, 2).reshape(bs, c, h, w), att_map

    def forward(self, x):
        x = self.conv1(x)
        x = self.scale * self.use_attention(x)[0] + x
        x = F.relu(x)
        x = F.relu(self.bn1(self.conv2(x)))
        x = F.relu(self.bn2(self.conv3(x)))
        x = x.view(x.shape[0], -1)
        x = F.relu(self.fc_1(x))
        return self.fc_out(x)


        
        

In [31]:
# Set device to GPU_indx if GPU is avaliable
gpu_indx = 0
device = torch.device(gpu_indx if torch.cuda.is_available() else 'cpu')

In [32]:
dataiter = next(iter(test_loader))

test_sample, test_label = dataiter

In [33]:
model = CNN(channels_in=test_sample.shape[1], y_size=test_label.shape[1]).to(device)

In [34]:
num_model_params = 0
for param in model.parameters():
    num_model_params += param.flatten().shape[0]

print("-This Model Has %d (Approximately %d Million) Parameters!" % (num_model_params, num_model_params//1e6))

-This Model Has 6565735 (Approximately 6 Million) Parameters!


In [35]:
# Pass image through network
out = model(test_sample.to(device))
# Check output
out.shape

torch.Size([64, 38])

In [36]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [37]:
def train(data_loader, model, loss_fn, optimizer):
    size = len(data_loader.dataset)
    model.train()
    with torch.enable_grad():
        for batch, (X,y) in enumerate(data_loader):
            X, y = X.to(device), y.to(device)

            pred = model(X)
            loss = loss_fn(pred, y)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if batch % 10 == 0:
                loss, current = loss.item(), (batch) * len(X)
                print(f"Loss {loss:>7f} [{current:>5d}/{size:>5d}]")

In [38]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
    test_loss /= num_batches
    print(f"Test Error: \n Avg loss : {test_loss:>8f} \n")

# CNN-ATN WITH OUT PHYSIC--INFORMED

In [39]:
# epochs = 100
# for t in range(epochs):
#     print(f"Epoch {t + 1}\n -------------------")
#     train(train_loader, model, loss_fn, optimizer)
#     test(test_loader, model, loss_fn)
# print("Done !!")

# CNN-ATN-WITH-PHYSIC-INFORMED

In [40]:
from pypower import idx_bus, idx_gen, idx_brch
from pypower.api import makeYbus, ext2int

net = ext2int(net)

In [41]:
class OPF_metric():
    def __init__(self, net):
        basemva = net['baseMVA']
        linear_cost = net['gencost'][:,5]
        self.cost_coef = torch.tensor(linear_cost).to(torch.float32)

        pg_max = net['gen'][:, idx_gen.PMAX] / basemva
        pg_min = net['gen'][:, idx_gen.PMIN] / basemva
        qg_max = net['gen'][:, idx_gen.QMAX] / basemva
        qg_min = net['gen'][:, idx_gen.QMIN] / basemva
        vm_max = net['bus'][:, idx_bus.VMAX]
        vm_min = net['bus'][:, idx_bus.VMIN]
        va_max = [np.pi/2 for i in range(len(net['bus']))]
        va_min = [-np.pi/2 for i in range(len(net['bus']))]
        outputs_min = np.concatenate([pg_min, qg_min, vm_min, va_min])
        outputs_max = np.concatenate([pg_max, qg_max, vm_max, va_max])
        self.outputs_min = torch.as_tensor(outputs_min).to(torch.float32).view(1,-1)
        self.outputs_max = torch.as_tensor(outputs_max).to(torch.float32).view(1,-1)

        # ============= Calculate the bus admittance matrix (Ybus) and branch admittance matrices (Yf, Yt) ============#
        net = ext2int(net)
        Ybus, Yf, Yt = makeYbus(net['baseMVA'], net['bus'], net['branch'])
        Ybus = Ybus.todense()
        self.Ybus_real = torch.as_tensor(Ybus.real).to(torch.float32)
        self.Ybus_imag = torch.as_tensor(Ybus.imag).to(torch.float32)
        Yf = Yf.todense()
        Yt = Yt.todense()
        self.Yf_real = torch.as_tensor(Yf.real).to(torch.float32)
        self.Yf_imag = torch.as_tensor(Yf.imag).to(torch.float32)
        self.Yt_real = torch.as_tensor(Yt.real).to(torch.float32)
        self.Yt_imag = torch.as_tensor(Yt.imag).to(torch.float32)
        
        self.gen_bus_index = net['gen'][:,idx_gen.GEN_BUS]
        self.load_bus_index = [i for i in range(14) if net['bus'][i, idx_bus.PD]>0]

        self.fbus = net['branch'][:,idx_brch.F_BUS].astype(int)
        self.tbus = net['branch'][:,idx_brch.T_BUS].astype(int)
        self.smax = torch.as_tensor(net['branch'][:,idx_brch.RATE_A] / basemva).to(torch.float32)
        self.angmax = torch.as_tensor(np.deg2rad(net['branch'][:,idx_brch.ANGMAX])).to(torch.float32)

    # Let's find the generation cost
    def cal_gen_cost(self, outputs):
        device = outputs.device  # Get the device of the outputs tensor
        cost_coef = self.cost_coef.to(device)
        pg = outputs[:, :5]
        return (pg  * cost_coef).sum(1)

    def cal_upper_lower_bound_violation(self, outputs):
        device = outputs.device  # Get the device of the outputs tensor
        outputs_max = self.outputs_max.to(device)  # Move to the same device
        outputs_min = self.outputs_min.to(device)  # Move to the same device
    
        # Use the local variables outputs_max and outputs_min
        vio_1 = torch.relu(outputs - outputs_max)
        vio_2 = torch.relu(outputs_min - outputs)
        return vio_1 + vio_2

    def gen_load_to_bus(self, inputs, outputs):
        device = outputs.device
        batch = inputs.shape[0]
        # Extract diagonal non-zero values of pd and qd from inputs
        pd_list = [torch.diagonal(inputs[b, 0, :, :], dim1=-2, dim2=-1)[torch.diagonal(inputs[b, 0, :, :], dim1=-2, dim2=-1) != 0] for b in range(batch)]
        qd_list = [torch.diagonal(inputs[b, 1, :, :], dim1=-2, dim2=-1)[torch.diagonal(inputs[b, 1, :, :], dim1=-2, dim2=-1) != 0] for b in range(batch)]
        # Pad sequences to make them the same size
        pd = torch.nn.utils.rnn.pad_sequence(pd_list, batch_first=True).to(device)
        qd = torch.nn.utils.rnn.pad_sequence(qd_list, batch_first=True).to(device)
    
        pg = outputs[:, 0:5]
        qg = outputs[:, 5:10]
        bus_pg = torch.zeros([batch, 14], device=device)
        bus_qg = torch.zeros([batch, 14], device=device)
        for i, bus_index in enumerate(self.gen_bus_index):
            # print(bus_index)
            bus_pg[:, int(bus_index)] = bus_pg[:, int(bus_index)] + pg[:, i]
            bus_qg[:, int(bus_index)] = bus_qg[:, int(bus_index)] + qg[:, i]
        bus_pd = torch.zeros([batch, 14], device=device)
        bus_qd = torch.zeros([batch, 14], device=device)
        for i, bus_index in enumerate(self.load_bus_index):
            bus_pd[:, bus_index] = bus_pd[:, bus_index] + pd[:, i]
            bus_qd[:, bus_index] = bus_qd[:, bus_index] + qd[:, i]
        return bus_pg, bus_qg, bus_pd, bus_qd

    def cal_power_balance_violation(self, inputs, outputs):
        device = outputs.device 
        bus_pg, bus_qg, bus_pd, bus_qd = self.gen_load_to_bus(inputs, outputs)
        bus_p_inj = bus_pg - bus_pd
        bus_q_inj = bus_qg - bus_qd
        
        vm, va = outputs[:, 10:24], outputs[:, 24:38]
        vr = vm * torch.cos(va)
        vi = vm * torch.sin(va)
        
        self.Ybus_real = self.Ybus_real.to(device)
        self.Ybus_imag = self.Ybus_imag.to(device)

        Ir = torch.matmul(vr, self.Ybus_real) - vi @ self.Ybus_imag
        Ii = vi @ self.Ybus_real + vr @ self.Ybus_imag
        bus_p = (vr * Ir) + (vi * Ii)
        bus_q = (vi * Ir) - (vr * Ii)
        return torch.abs(bus_p_inj - bus_p), torch.abs(bus_q_inj - bus_q)

    # Let's calculate the branch flow & angle constraint violation
    def cal_branch_flow_vio(self, outputs):
        device = outputs.device 
        vm, va = outputs[:, 10:24], outputs[:, 24:38]
        vr = vm * torch.cos(va)
        vi = vm * torch.sin(va)
        
        self.Yf_real = self.Yf_real.to(device)
        self.Yf_imag = self.Yf_imag.to(device)
        self.Yt_real = self.Yt_real.to(device)
        self.Yt_imag = self.Yt_imag.to(device)
        self.smax = self.smax.to(device)
        self.angmax = self.angmax.to(device)
        
        Irf = vr @ self.Yf_real.T - vi @ self.Yf_imag.T
        Iif = vi @ self.Yf_real.T + vr @ self.Yf_imag.T
        branch_pf = vr[:, self.fbus] * Irf + vi[:, self.fbus] * Iif
        branch_qf = vi[:, self.fbus] * Irf - vr[:, self.fbus] * Iif
        sf = torch.sqrt((torch.square(branch_pf) + torch.square(branch_qf)))

        Irt = vr @ self.Yt_real.T - vi @ self.Yt_imag.T
        Iit = vi @ self.Yt_real.T + vr @ self.Yt_imag.T
        branch_pf = vr[:, self.tbus] * Irt + vi[:, self.tbus] * Iit
        branch_qf = vi[:, self.tbus] * Irt - vr[:, self.tbus] * Iit
        st = torch.sqrt((torch.square(branch_pf) + torch.square(branch_qf)))

        branch_flow = torch.maximum(sf, st)
        branch_ang = torch.abs(va[:, self.fbus] - va[:, self.tbus])
        return torch.relu(branch_flow - self.smax), torch.relu(branch_ang - self.angmax)

In [42]:
opf_metric = OPF_metric(net)

In [43]:
def test(dataloader, model, opf_metric):
    device = next(model.parameters()).device
    batch_snapshot = next(iter(dataloader))
    X, Y = batch_snapshot[0].to(device), batch_snapshot[1].to(device)
    with torch.no_grad():
      Y_pred = model(X)

    test_sol_mse = ((Y_pred - Y)**2)

    Y_pred_cost = opf_metric.cal_gen_cost(Y_pred)
    Y_opt_cost = opf_metric.cal_gen_cost(Y)
    test_obj_gap = (Y_pred_cost - Y_opt_cost)/Y_opt_cost

    vio_bound = opf_metric.cal_upper_lower_bound_violation(Y_pred)
    vio_p_mis, vio_q_mis = opf_metric.cal_power_balance_violation(X, Y_pred)
    vio_flow, vio_ang = opf_metric.cal_branch_flow_vio(Y_pred)
    eq_vio = torch.cat([vio_p_mis, vio_q_mis], dim=1)
    ineq_vio = torch.cat([vio_bound, vio_flow, vio_ang], dim=1)
    return test_sol_mse, test_obj_gap, eq_vio, ineq_vio

def model_train_test(train_dataloader, test_dataloader, model,
                     loss_fn, opf_metric, epochs = 10, penalty=False):
    torch.manual_seed(42)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    results = {
        "train_losses": [],
        "test_mse": [],
        "opt_gaps": [],
        "eq_vios": [],
        "ineq_vios": []
    }
    print(f"==============================================Start training on {device}===========================================")

    for t in range(epochs):
        model.train()
        train_loss = []
        with torch.enable_grad():
            for batch, (X, y) in enumerate(train_dataloader):
                X, y = X.to(device), y.to(device)
                # Compute prediction error
                pred = model(X)
                if penalty:
                  loss = loss_fn(X, pred, y)
                else:
                  loss = loss_fn(pred, y)
                # Backpropagation
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                train_loss.append(loss.item())
            train_loss = np.mean(train_loss)
            results["train_losses"].append(train_loss)
            test_sol_mse, test_obj_gap, eq_vio, ineq_vio = test(test_dataloader, model, opf_metric)
            
            results["test_mse"].append(test_sol_mse)
            results["opt_gaps"].append(test_obj_gap)
            results["eq_vios"].append(eq_vio)
            results["ineq_vios"].append(ineq_vio)
            
            print(f"Epoch {t+1} | Train loss: {train_loss:.7f} | ",
                f"Test mse: {test_sol_mse.mean():.7f} | ",
                f"opt gap: {test_obj_gap.mean():.7f}% | ",
                f"eq vio: {eq_vio.sum(1).mean():.7f} | ",
                f"ineq vio: {ineq_vio.sum(1).mean():.7f} |")
        
    return results


In [50]:
model_base = CNN(channels_in=test_sample.shape[1], y_size=test_label.shape[1]).to(device)

In [51]:
results = model_train_test(train_loader, test_loader, model_base, nn.MSELoss(), opf_metric, epochs=100)

==============================================Start training on cuda:0===========================================
Epoch 1 | Train loss: 0.0346494 |  Test mse: 0.0054550 |  opt gap: 0.0471843% |  eq vio: 12.3131046 |  ineq vio: 0.0522478 |
Epoch 2 | Train loss: 0.0052771 |  Test mse: 0.0072518 |  opt gap: -0.0654350% |  eq vio: 17.5149212 |  ineq vio: 0.2588098 |
Epoch 3 | Train loss: 0.0057778 |  Test mse: 0.0061103 |  opt gap: 0.0447167% |  eq vio: 15.3438139 |  ineq vio: 0.1523820 |
Epoch 4 | Train loss: 0.0057047 |  Test mse: 0.0057441 |  opt gap: -0.0405603% |  eq vio: 13.9810009 |  ineq vio: 0.1112424 |
Epoch 5 | Train loss: 0.0053968 |  Test mse: 0.0054537 |  opt gap: -0.0240570% |  eq vio: 14.2491493 |  ineq vio: 0.1629013 |
Epoch 6 | Train loss: 0.0142118 |  Test mse: 0.0051813 |  opt gap: 0.0323735% |  eq vio: 14.5456505 |  ineq vio: 0.1260502 |
Epoch 7 | Train loss: 0.0050967 |  Test mse: 0.0044392 |  opt gap: 0.1009904% |  eq vio: 12.1209908 |  ineq vio: 0.1305413 |
Epoch 8 

In [46]:
""" Let's add the violatio into loss function and minimize it
"""
class MSEPenaltyLoss(nn.Module):
    def __init__(self, opf_metric, w_eq=0.1, w_ineq=0.1):
        super().__init__()
        self.w_eq = w_eq
        self.w_ineq = w_ineq
        self.mse = nn.MSELoss()
        self.opf_metric = opf_metric

    def forward(self, X, pred, target):
        MSE = self.mse(pred, target)
        # print(f"MSE: {MSE.item()}")
    
        vio_bound = self.opf_metric.cal_upper_lower_bound_violation(pred)
        # print(f"Vio_bound max: {vio_bound.max().item()}, min: {vio_bound.min().item()}")
    
        vio_p_mis, vio_q_mis = self.opf_metric.cal_power_balance_violation(X, pred)
        # print(f"Vio_p_mis max: {vio_p_mis.max().item()}, min: {vio_p_mis.min().item()}")
        # print(f"Vio_q_mis max: {vio_q_mis.max().item()}, min: {vio_q_mis.min().item()}")
    
        vio_flow, vio_ang = self.opf_metric.cal_branch_flow_vio(pred)
        # print(f"Vio_flow max: {vio_flow.max().item()}, min: {vio_flow.min().item()}")
        # print(f"Vio_ang max: {vio_ang.max().item()}, min: {vio_ang.min().item()}")
    
        eq_vio = torch.cat([vio_p_mis, vio_q_mis], dim=1)
        ineq_vio = torch.cat([vio_bound, vio_flow, vio_ang], dim=1)
    
        loss = MSE + self.w_eq * eq_vio.mean() + self.w_ineq * ineq_vio.mean()
        # print(f"Loss: {loss.item()}")
        return loss


In [47]:
class BoundCNN(nn.Module):
    def __init__(self, channels_in, y_size, yl, yu):
        super(BoundCNN, self).__init__()
        self.conv1 = nn.Conv2d(channels_in, 64, 3, 1, 1, padding_mode='reflect')

        self.norm = nn.LayerNorm(64)
        self.mha  = nn.MultiheadAttention(64, num_heads=1, batch_first=True)
        self.scale = nn.Parameter(torch.zeros(1))

        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.bn1 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(128)

        self.drop = nn.Dropout(0.5)
        self.fc_1 = nn.Linear(128*14*14, 256)
        self.fc_out = nn.Linear(256, y_size)

        self.register_buffer('yl', yl.clone().detach().float())
        self.register_buffer('yu', yu.clone().detach().float())

    def use_attention(self, x):
        bs, c, h, w = x.shape
        x_attn = x.reshape(bs, c, h * w).transpose(1, 2)    # BS X HW X C

        x_attn = self.norm(x_attn)
        att_out, att_map = self.mha(x_attn, x_attn, x_attn)
        return att_out.transpose(1, 2).reshape(bs, c, h, w), att_map

    def forward(self, x):
        x = self.conv1(x)
        x = self.scale * self.use_attention(x)[0] + x
        x = F.relu(x)
        x = F.relu(self.bn1(self.conv2(x)))
        x = F.relu(self.bn2(self.conv3(x)))
        x = x.view(x.shape[0], -1)
        x = F.relu(self.fc_1(x))
        x = self.fc_out(x)
        x = torch.sigmoid(x)
        x = x * (self.yu - self.yl) + self.yl
        return x

In [52]:
model_base = BoundCNN(channels_in=test_sample.shape[1], y_size=test_label.shape[1], yl=opf_metric.outputs_min, yu=opf_metric.outputs_max).to(device)

In [53]:
results = model_train_test(train_loader, test_loader, model_base,MSEPenaltyLoss(opf_metric), opf_metric, epochs=1000, penalty=True)

==============================================Start training on cuda:0===========================================
Epoch 1 | Train loss: 0.3645513 |  Test mse: 0.0068952 |  opt gap: 0.0247834% |  eq vio: 14.8093138 |  ineq vio: 0.0000000 |
Epoch 2 | Train loss: 0.0521783 |  Test mse: 0.0063355 |  opt gap: 0.0288383% |  eq vio: 10.5737066 |  ineq vio: 0.0000000 |
Epoch 3 | Train loss: 0.0260041 |  Test mse: 0.0056034 |  opt gap: 0.0309708% |  eq vio: 4.2569170 |  ineq vio: 0.0000000 |
Epoch 4 | Train loss: 0.0237443 |  Test mse: 0.0055558 |  opt gap: 0.0356979% |  eq vio: 4.8189101 |  ineq vio: 0.0000000 |
Epoch 5 | Train loss: 0.0179920 |  Test mse: 0.0053783 |  opt gap: 0.0346881% |  eq vio: 3.0243824 |  ineq vio: 0.0000000 |
Epoch 6 | Train loss: 0.0147819 |  Test mse: 0.0053441 |  opt gap: 0.0246283% |  eq vio: 3.4222517 |  ineq vio: 0.0000000 |
Epoch 7 | Train loss: 0.0147384 |  Test mse: 0.0053151 |  opt gap: 0.0279695% |  eq vio: 2.7407727 |  ineq vio: 0.0000000 |
Epoch 8 | Train 